In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, os
import matplotlib.pyplot as plt
import urllib.request
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

# ==========================================
# 0. Setup
# ==========================================

def drawing_to_stroke3(drawing):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

def normalise_stroke3(stroke3):
    s = stroke3.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d  = json.loads(line)
                s3 = drawing_to_stroke3(d['drawing'])
                s3 = normalise_stroke3(s3)
                if len(s3) > max_len: s3 = s3[:max_len]
                self.samples.append(torch.tensor(s3, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

class Encoder(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm      = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu     = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[0], h[1]], dim=-1)
        return self.fc_mu(h), self.fc_logvar(h)

def reparameterise(mu, logvar):
    std = torch.exp(0.5 * logvar)
    return mu + torch.randn_like(std) * std

def kl_loss(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f'Dataset: {len(dataset)} samples')
print('Setup complete.')



In [ ]:

# ==========================================
# 1. The Decoder
# ==========================================

class Decoder(nn.Module):
    def __init__(self, latent_dim=128, hidden_dim=512, output_dim=3):
        super().__init__()
        self.fc_hidden = nn.Linear(latent_dim, hidden_dim)
        self.fc_cell   = nn.Linear(latent_dim, hidden_dim)
        self.lstm = nn.LSTM(input_size=output_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, target_seq):
        batch_size = z.shape[0]
        h0 = torch.tanh(self.fc_hidden(z)).unsqueeze(0)   # (1, batch, hidden)
        c0 = torch.tanh(self.fc_cell(z)).unsqueeze(0)

        start_token = torch.zeros(batch_size, 1, 3, device=z.device)
        decoder_input = torch.cat([start_token, target_seq[:, :-1, :]], dim=1)

        output, _ = self.lstm(decoder_input, (h0, c0))
        output = self.fc_out(output)   # (batch, seq_len, 3)
        return output

    def generate(self, z, max_len=200):
        batch_size = z.shape[0]
        h = torch.tanh(self.fc_hidden(z)).unsqueeze(0)
        c = torch.tanh(self.fc_cell(z)).unsqueeze(0)

        input_step = torch.zeros(batch_size, 1, 3, device=z.device)
        outputs = []

        for _ in range(max_len):
            out, (h, c) = self.lstm(input_step, (h, c))
            step = self.fc_out(out)           # (batch, 1, 3)
            outputs.append(step)
            input_step = step                 # feed prediction back

        return torch.cat(outputs, dim=1)      # (batch, max_len, 3)

decoder = Decoder(latent_dim=128, hidden_dim=512)
print(decoder)

# TODO: Why do we use teacher forcing during training but not during generation?
#       What could go wrong if we used the model's own predictions during training?

# OBSERVATIONS:
# 1. Teacher forcing stabilizes training by feeding the true historical coordinates at step (t-1) 
#    to predict step t. This avoids compounding cascade errors. During generation, we lack ground-truth reference labels, 
#    forcing the model to be truly autoregressive.
# 2. Without teacher forcing during early training, a single bad mistake at step 1 would propagate, 
#    making the entire subsequent path completely incoherent. The model would spend hours learning to correct 
#    its own accumulated junk errors rather than learning the actual underlying geometry of a cat.



In [ ]:

# ==========================================
# 2. The Full VAE
# ==========================================

class SketchVAE(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=256, latent_dim=128, decoder_hidden=512):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, decoder_hidden, input_dim)

    def forward(self, x, lengths):
        mu, logvar = self.encoder(x, lengths)
        z          = reparameterise(mu, logvar)
        recon      = self.decoder(z, x)
        return recon, mu, logvar

model = SketchVAE()
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

# Quick forward pass test
padded, lengths = next(iter(loader))
recon, mu, logvar = model(padded, lengths)
print('Input shape      :', padded.shape)
print('Reconstruction   :', recon.shape)
print('mu shape         :', mu.shape)



In [ ]:

# ==========================================
# 3. The Loss Function
# ==========================================

def vae_loss(recon, target, mu, logvar, kl_weight=0.5):
    recon_pos  = F.mse_loss(recon[:, :, :2], target[:, :, :2])
    pen_pred   = torch.sigmoid(recon[:, :, 2])
    recon_pen  = F.binary_cross_entropy(pen_pred, target[:, :, 2], reduction='mean')
    kl         = kl_loss(mu, logvar)
    total      = recon_pos + recon_pen + kl_weight * kl
    return total, recon_pos, recon_pen, kl

loss, rp, rpen, kl = vae_loss(recon, padded, mu, logvar)
print(f'Total loss  : {loss.item():.4f}')
print(f'Recon pos   : {rp.item():.4f}')
print(f'Recon pen   : {rpen.item():.4f}')
print(f'KL loss     : {kl.item():.4f}')



In [ ]:

# ==========================================
# 4. Training Loop
# ==========================================

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = SketchVAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
history    = {'total': [], 'recon_pos': [], 'recon_pen': [], 'kl': []}

for epoch in range(num_epochs):
    model.train()
    epoch_losses = {'total': 0, 'recon_pos': 0, 'recon_pen': 0, 'kl': 0}
    kl_weight = min(0.5, epoch / num_epochs)

    for padded, lengths in loader:
        padded = padded.to(device)
        recon, mu, logvar = model(padded, lengths)
        loss, rp, rpen, kl = vae_loss(recon, padded, mu, logvar, kl_weight)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_losses['total']     += loss.item()
        epoch_losses['recon_pos'] += rp.item()
        epoch_losses['recon_pen'] += rpen.item()
        epoch_losses['kl']        += kl.item()

    n = len(loader)
    for k in history:
        history[k].append(epoch_losses[k] / n)

    print(f'Epoch {epoch+1:02d}/{num_epochs} | '
          f'loss {history["total"][-1]:.4f} | '
          f'recon_pos {history["recon_pos"][-1]:.4f} | '
          f'kl {history["kl"][-1]:.4f} | '
          f'kl_weight {kl_weight:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(history['total']);     axes[0].set_title('Total loss')
axes[1].plot(history['recon_pos']); axes[1].set_title('Reconstruction loss (position)')
axes[2].plot(history['kl']);        axes[2].set_title('KL loss')
for ax in axes:
    ax.set_xlabel('Epoch')
plt.tight_layout()
plt.show()



In [ ]:

# ==========================================
# 5. Visualising Reconstructions
# ==========================================

def stroke3_to_absolute(stroke3):
    abs_coords = np.cumsum(stroke3[:, :2], axis=0)
    pen_lifted  = stroke3[:, 2]
    return abs_coords, pen_lifted

def plot_sketch(stroke3_np, title='', ax=None):
    coords, pen_lifted = stroke3_to_absolute(stroke3_np)
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off'); ax.set_title(title, fontsize=9)
    start = 0
    for i in range(len(pen_lifted)):
        if pen_lifted[i] > 0.5:
            seg = coords[start : i + 1]
            ax.plot(seg[:, 0], seg[:, 1], 'k-', linewidth=1.5)
            start = i + 1

model.eval()
with torch.no_grad():
    sample_batch, sample_lengths = next(iter(loader))
    sample_batch = sample_batch.to(device)
    recon, mu, logvar = model(sample_batch, sample_lengths)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    original = sample_batch[i].cpu().numpy()
    reconstructed = recon[i].cpu().numpy()
    plot_sketch(original,      title=f'Original {i}',       ax=axes[0, i])
    plot_sketch(reconstructed, title=f'Reconstructed {i}',  ax=axes[1, i])

plt.suptitle('Top: original   Bottom: reconstructed', fontsize=11)
plt.tight_layout()
plt.show()



In [ ]:

# ==========================================
# 6. Generating New Sketches from Random Z
# ==========================================

model.eval()
with torch.no_grad():
    z_random = torch.randn(8, 128).to(device)
    generated = model.decoder.generate(z_random, max_len=150)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    plot_sketch(generated[i].cpu().numpy(), title=f'Generated {i}', ax=ax)

plt.suptitle('Sketches generated from random z', fontsize=11)
plt.tight_layout()
plt.show()

# TODO: Do the generated sketches look like cats? If not, why might that be?
#       What would you change to improve the quality?

# OBSERVATIONS:
# 1. They often look messy or unrecognizable compared to true reconstructions. This is because standard autoregressive 
#    LSTM decoders suffer from error accumulation during long generation phases, or exposure bias. 
#    Without a structural guide, the point-by-point deltas drift completely out of control.
# 2. To fix this, we should upgrade to a hierarchical decoder architecture (like Sketch-RNN). Instead of forcing 
#    a single LSTM to manage long sequences, a higher-level "conductor" network should output sub-commands to coordinate 
#    separate "worker" stroke generators.



In [ ]:

# ==========================================
# 7. Latent Space Interpolation
# ==========================================

model.eval()
with torch.no_grad():
    s1 = dataset[0].unsqueeze(0).to(device)
    s2 = dataset[1].unsqueeze(0).to(device)
    mu1, _ = model.encoder(s1, [s1.shape[1]])
    mu2, _ = model.encoder(s2, [s2.shape[1]])

    steps = 6
    alphas = torch.linspace(0, 1, steps)
    z_interp = torch.stack([alpha * mu2 + (1 - alpha) * mu1 for alpha in alphas]).squeeze(1)
    generated = model.decoder.generate(z_interp, max_len=150)

fig, axes = plt.subplots(1, steps, figsize=(3 * steps, 3))
for i, ax in enumerate(axes):
    plot_sketch(generated[i].cpu().numpy(), title=f'alpha={alphas[i]:.1f}', ax=ax)

plt.suptitle('Latent interpolation between two cat sketches', fontsize=11)
plt.tight_layout()
plt.show()

# TODO: Does the interpolation look smooth? What does smoothness here mean for the model?

# OBSERVATIONS:
# 1. The interpolation displays gradual modification of structural characteristics (e.g., ear proportions warping or face bounds expanding).
# 2. Smoothness signifies that the latent space topology is dense, organized, and structurally logical. It demonstrates that 
#    the KL loss successfully penalized isolated feature groupings, encouraging the model to map structural properties 
#    onto continuous geometric vector trajectories.



In [ ]:

# ==========================================
# 8. Save the Model
# ==========================================

torch.save(model.state_dict(), 'sketch_vae.pth')
print('Model saved to sketch_vae.pth')
print('Checkpoint 2 complete -- Latent Compression!')